<a href="https://colab.research.google.com/github/moeeed2006-ops/Abdul-Moeed-flyrank-ml-work/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moeeed2006-ops/Abdul-Moeed-flyrank-ml-work/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

# Load dataset or generate fallback data matching FlyRank schema
data_paths = ['../data/flyrank_dataset.csv', 'work/data/flyrank_dataset.csv', 'flyrank_dataset.csv']
df = None

for path in data_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Loaded dataset from {path}")
        break

if df is None:
    print("Generating synthetic dataset for playbook export...")
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'url_id': [f'url_{i:04d}' for i in range(n)],
        'days_since_last_refresh': np.random.randint(1, 365, n),
        'ctr_position_gap': np.random.uniform(-0.05, 0.20, n),
        'impressions': np.random.randint(50, 50000, n),
        'current_ctr': np.random.uniform(0.01, 0.15, n)
    })

print(f"Dataset ready with {len(df)} rows.")

Generating synthetic dataset for playbook export...
Dataset ready with 1000 rows.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Archetype-to-Action Mapping

| Archetype | Trigger Condition | Reason Code | Recommended Action |
|---|---|---|---|
| **Stale Underperformer** | Staleness >180 days & High CTR Deficit | `STALE_LOW_CTR` | `FULL_CONTENT_REFRESH` |
| **Title Under-Index** | High Impressions & High CTR Deficit | `HIGH_IMP_LOW_CTR` | `OPTIMIZE_TITLE_META` |
| **Decaying Asset** | Staleness >270 days & Dropping Traffic | `DECAY_TRAFFIC_DROP` | `UPDATE_FACTS_INTERNAL_LINKS` |
| **Low-Yield Content** | Low Impressions & Staleness >200 days | `LOW_VOL_STALE` | `MONITOR_OR_CONSOLIDATE` |

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Compute Playbook Scores and Map Reason Codes
df['playbook_score'] = (0.6 * (df['days_since_last_refresh'] / 365.0)) + (0.4 * (df['ctr_position_gap'] / 0.20))

def assign_playbook_action(row):
    if row['days_since_last_refresh'] > 180 and row['ctr_position_gap'] > 0.10:
        return 'STALE_LOW_CTR', 'FULL_CONTENT_REFRESH'
    elif row['ctr_position_gap'] > 0.10:
        return 'HIGH_IMP_LOW_CTR', 'OPTIMIZE_TITLE_META'
    elif row['days_since_last_refresh'] > 270:
        return 'DECAY_TRAFFIC_DROP', 'UPDATE_FACTS_INTERNAL_LINKS'
    else:
        return 'LOW_VOL_STALE', 'MONITOR_OR_CONSOLIDATE'

actions_and_reasons = df.apply(assign_playbook_action, axis=1)
df['reason_code'] = [ar[0] for ar in actions_and_reasons]
df['action_label'] = [ar[1] for ar in actions_and_reasons]

# Generate Decay/Refresh Insight Figure
plt.figure(figsize=(8, 4))
plt.scatter(df['days_since_last_refresh'], df['playbook_score'], alpha=0.5, c='#1f77b4')
plt.axvline(180, color='red', linestyle='--', label='180-Day Decay Threshold')
plt.title('Content Decay vs. Playbook Action Score')
plt.xlabel('Days Since Last Refresh')
plt.ylabel('Playbook Action Score')
plt.legend()
plt.tight_layout()

figure_path = '../figures/content_decay_analysis.png'
plt.savefig(figure_path, dpi=300)
plt.close()
print(f"Decay insight figure committed to: {figure_path}")

Decay insight figure committed to: ../figures/content_decay_analysis.png


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use and Operating Limits

- **Primary Intended Use:** Serves as an analytical decision-support tool for editorial and SEO content teams to prioritize monthly content updates.
- **Operating Scope:** Designed strictly for non-production batch ranking. Scores reflect potential optimization value, not guaranteed traffic improvements.
- **Cost/Value Threshold:** High-effort actions (`FULL_CONTENT_REFRESH`) should only be executed on pages with >1,000 monthly impressions to ensure favorable ROI relative to editor time.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Protocols & The No-Go List

#### Human Review Requirements
1. **Sanity Verification:** All recommendations scored in the top 10% must be reviewed by a human editor before any content modification.
2. **Intent Matching:** Confirm search intent has not fundamentally shifted before updating headings or titles.

#### The No-Go List (STRICTLY DO NOT AUTOMATE)
- **Legal & Compliance Pages:** Privacy policies, terms of service, and regulatory disclosures must never be automated or batch-updated.
- **Brand Pillar Pages:** Core brand landing pages and executive statements require manual editorial sign-off.
- **High-Converting Seasonal Pages:** Active seasonal campaigns must be excluded from automated refresh queues.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Light Monitoring & Retrain Triggers

- **Data Drift Trigger:** If the mean distribution of `ctr_position_gap` shifts by more than 15% across quarterly site crawls, re-calibrate score weights.
- **Performance Retrain Trigger:** Re-fit score thresholds bi-annually or whenever major search engine core layout updates occur.
- **Feedback Loop Integration:** Log human editor acceptance rates; if rejection rates exceed 20% for a specific reason code, trigger rule adjustment.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Export Ranked Action Queue CSV
queue_export = df[['url_id', 'playbook_score', 'reason_code', 'action_label']].sort_values(
    by='playbook_score', ascending=False
)
queue_path = '../outputs/content_action_playbook_queue.csv'
queue_export.to_csv(queue_path, index=False)

# 2. Export Playbook Summary Metrics JSON
playbook_summary = {
    'total_urls_ranked': len(df),
    'action_breakdown': df['action_label'].value_counts().to_dict(),
    'reason_code_breakdown': df['reason_code'].value_counts().to_dict(),
    'figure_exported': os.path.exists('../figures/content_decay_analysis.png')
}

json_path = '../outputs/w07_playbook_summary.json'
with open(json_path, 'w') as f:
    json.dump(playbook_summary, f, indent=2)

print(f"[✓] Ranked queue exported to: {queue_path}")
print(f"[✓] Playbook metrics receipt exported to: {json_path}")

[✓] Ranked queue exported to: ../outputs/content_action_playbook_queue.csv
[✓] Playbook metrics receipt exported to: ../outputs/w07_playbook_summary.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.